# Phase 3B: Building Energy Forecasting Models
## Building 59 South Wing — Stage 2 Forecasting ($t \to t+1$)

This notebook implements, benchmarks, and analyzes candidate machine learning models for:
1. **Primary Target**: Total South Wing electricity load (`south_wing_total_kwh_next_hour`)
2. **Oracle vs Chained Forecasting Regimes**: Quantifying real-world error propagation when driven by predicted occupancy.
3. **Submeter Diagnostic Models**: HVAC, Lighting, and Plug Load predictability.

### Methodology & Constraints:
- Chronological train/validation/test partitions are strictly observed.
- Energy predictors incorporate autoregressive energy lags, weather thermodynamics, candidate HVAC controls, and Stage 1 occupancy state.
- Chained inference feeds out-of-sample Stage 1 occupancy predictions directly into the energy forecaster.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on path
PROJ_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

from src.energy_model import (
    load_and_split_energy_data,
    train_and_eval_energy_regressors,
    evaluate_chained_energy,
    load_energy_model_artifacts
)
from src.occupancy_model import load_occupancy_model_artifacts
from src.evaluation import (
    compute_regression_metrics,
    plot_actual_vs_predicted_series,
    plot_residuals,
    plot_feature_importance,
    plot_oracle_vs_chained
)

DATA_PATH = PROJ_ROOT / 'data' / 'processed' / 'joint_modeling_data.parquet'
OCC_MODEL_DIR = PROJ_ROOT / 'models' / 'occupancy'
ENERGY_MODEL_DIR = PROJ_ROOT / 'models' / 'energy'


## 1. Load Data & Stage 1 Occupancy Model

In [2]:
energy_data = load_and_split_energy_data(DATA_PATH)
occ_artifact = load_occupancy_model_artifacts(OCC_MODEL_DIR)

occ_model = occ_artifact['model']
occ_scaler = occ_artifact['scaler']
occ_threshold = occ_artifact['threshold']

print(f"Energy Predictor Count : {len(energy_data['feature_cols'])}")
print(f"Train samples          : {len(energy_data['train']['X'])}")
print(f"Val samples            : {len(energy_data['val']['X'])}")
print(f"Test samples           : {len(energy_data['test']['X'])}")


Energy Predictor Count : 55
Train samples          : 4063
Val samples            : 888
Test samples           : 994


## 2. Benchmark Candidate Energy Regressors (Validation Split)

In [3]:
# 1. Train Regressors on Train Split
energy_results = train_and_eval_energy_regressors(
    energy_data['train']['X'],
    energy_data['train']['y'],
    energy_data['val']['X'],
    energy_data['val']['y'],
    energy_data['feature_cols'],
    random_state=42
)

# 2. Generate Out-Of-Sample Occupancy Predictions on Validation Set
from src.occupancy_model import load_and_split_occupancy_data
occ_data = load_and_split_occupancy_data(DATA_PATH)
X_val_occ_in = occ_scaler.transform(occ_data['val']['X']) if occ_scaler else occ_data['val']['X'].values
val_pred_occ = (occ_model.predict_proba(X_val_occ_in)[:, 1] >= occ_threshold).astype(int)

# 3. Evaluate Oracle vs Chained Performance
energy_rows = []
for name, r in energy_results.items():
    m_oracle = r['metrics']
    chained_eval = evaluate_chained_energy(
        model=r['model'],
        X_eval=energy_data['val']['X'],
        y_eval=energy_data['val']['y'],
        predicted_occupancy=val_pred_occ,
        is_linear='Ridge' in name,
        scaler=r['scaler']
    )
    m_chained = chained_eval['metrics']
    energy_rows.append({
        'Model': name,
        'Oracle MAE': m_oracle['mae'],
        'Oracle RMSE': m_oracle['rmse'],
        'Oracle R2': m_oracle['r2'],
        'Chained MAE': m_chained['mae'],
        'Chained RMSE': m_chained['rmse'],
        'Chained R2': m_chained['r2'],
        'Delta MAE': round(m_chained['mae'] - m_oracle['mae'], 4)
    })

df_energy_val = pd.DataFrame(energy_rows).sort_values(by='Chained RMSE')
display(df_energy_val)


[DEPENDENCY WARNING] LightGBM is unavailable or failed runtime execution: exception: access violation reading 0x0000000000000000. Skipping and continuing with remaining models.


,Model,Oracle MAE,Oracle RMSE,Oracle R2,Chained MAE,Chained RMSE,Chained R2,Delta MAE
0,Ridge Regression,5.0815,6.2471,0.3899,5.0992,6.2754,0.3843,0.0177
1,Random Forest,6.2131,7.6022,0.0965,6.2125,7.6017,0.0966,-0.0006
2,Extra Trees,6.4731,7.9109,0.0216,6.4869,7.9315,0.0165,0.0138
3,XGBoost,6.5503,8.1174,-0.0301,6.5676,8.1460,-0.0374,0.0173


## 3. Final Test Evaluation: Champion Energy Forecaster

In [4]:
energy_artifact = load_energy_model_artifacts(ENERGY_MODEL_DIR)
champion_energy_model = energy_artifact['model']
champion_energy_scaler = energy_artifact['scaler']

print(f"Loaded Champion Energy Model: {energy_artifact['metadata']['model_name']}")

# Generate Out-Of-Sample Test Occupancy Predictions
X_test_occ_in = occ_scaler.transform(occ_data['test']['X']) if occ_scaler else occ_data['test']['X'].values
test_pred_occ = (occ_model.predict_proba(X_test_occ_in)[:, 1] >= occ_threshold).astype(int)

# Oracle Test Evaluation
X_test_oracle_in = champion_energy_scaler.transform(energy_data['test']['X']) if champion_energy_scaler else energy_data['test']['X'].values
test_oracle_preds = np.maximum(0.0, champion_energy_model.predict(X_test_oracle_in))
test_oracle_m = compute_regression_metrics(energy_data['test']['y'], test_oracle_preds)

# Chained Test Evaluation
test_chained_eval = evaluate_chained_energy(
    model=champion_energy_model,
    X_eval=energy_data['test']['X'],
    y_eval=energy_data['test']['y'],
    predicted_occupancy=test_pred_occ,
    is_linear='Ridge' in energy_artifact['metadata']['model_name'],
    scaler=champion_energy_scaler
)
test_chained_preds = test_chained_eval['predictions']
test_chained_m = test_chained_eval['metrics']

print(f"Oracle  Test Performance: MAE={test_oracle_m['mae']:.4f} kWh, RMSE={test_oracle_m['rmse']:.4f} kWh, R2={test_oracle_m['r2']:.4f}, MAPE={test_oracle_m['mape']}%")
print(f"Chained Test Performance: MAE={test_chained_m['mae']:.4f} kWh, RMSE={test_chained_m['rmse']:.4f} kWh, R2={test_chained_m['r2']:.4f}, MAPE={test_chained_m['mape']}%")
print(f"Error Propagation Delta: {test_chained_m['mae'] - test_oracle_m['mae']:+.4f} kWh")


Loaded Champion Energy Model: Ridge Regression
Oracle  Test Performance: MAE=7.7061 kWh, RMSE=9.8759 kWh, R2=-0.0723, MAPE=59.69%
Chained Test Performance: MAE=7.6733 kWh, RMSE=9.8357 kWh, R2=-0.0636, MAPE=59.56%
Error Propagation Delta: -0.0328 kWh


## 4. Visual Diagnostics & Error Analysis

In [5]:
# 1. Actual vs Predicted Series & Scatter
plot_actual_vs_predicted_series(
    energy_data['test']['y'],
    test_chained_preds,
    index=energy_data['test']['index'],
    title='Ridge Regression (Chained Test Forecast)'
)
plt.show()

# 2. Residual Distribution & Timeline
plot_residuals(
    energy_data['test']['y'],
    test_chained_preds,
    index=energy_data['test']['index'],
    title='Ridge Energy Forecast Residuals'
)
plt.show()

# 3. Oracle vs Chained Comparison
plot_oracle_vs_chained(
    energy_data['test']['y'],
    test_oracle_preds,
    test_chained_preds,
    index=energy_data['test']['index'],
    title='Oracle vs Chained Forecast Comparison'
)
plt.show()
